<a href="https://colab.research.google.com/github/Kayla-afk/Tugas-Kuliah-D4-Sains-Data-Terapan/blob/main/DM_M5_3324600023_Kayla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup dan Import Library

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

Load Dataset

In [2]:
import pandas as pd

dataset = pd.read_csv('/content/titanic.csv')
test_dataset = pd.read_csv('/content/titanic_test.csv')
test_label_df = pd.read_csv('/content/titanic_testlabel.csv')

# cek isi
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Penjelasan

**Tujuan:**
Memuat dataset yang digunakan dalam proses pelatihan dan pengujian model KNN.

**Proses:**
Kode mengimpor library `pandas` untuk manipulasi data, kemudian membaca tiga file CSV:
- `titanic.csv` sebagai data training
- `titanic_test.csv` sebagai data testing (fitur)
- `titanic_testlabel.csv` sebagai label data testing

Fungsi `pd.read_csv()` digunakan untuk mengonversi file CSV menjadi DataFrame.

Perintah `dataset.head()` digunakan untuk menampilkan 5 baris pertama dari dataset training sebagai langkah verifikasi awal.

**Output:**
- `dataset`: DataFrame berisi data training
- `test_dataset`: DataFrame berisi data uji (fitur)
- `test_label_df`: DataFrame berisi label data uji
- Tampilan 5 baris pertama dari `dataset`



Ambil Fitur (Age, Fare) + Handle Missing (Train)

In [3]:
# ambil fitur
train_data = dataset[['Age', 'Fare']]

# cek missing
print(train_data.isnull().sum())

# catat posisi missing
pos_missing_train = train_data[train_data.isnull().any(axis=1)].index

# hapus missing
train_data_clean = train_data.dropna()

print("Jumlah data train setelah cleaning:", len(train_data_clean))

Age     177
Fare      0
dtype: int64
Jumlah data train setelah cleaning: 714


Penjelasan:

**Tujuan:**
Memilih fitur yang relevan (Age dan Fare) serta menangani missing values pada data training agar siap digunakan dalam model KNN.

**Proses:**
Kode mengambil dua kolom, yaitu `Age` dan `Fare`, sebagai fitur utama.  
Selanjutnya dilakukan pengecekan jumlah missing values pada masing-masing kolom menggunakan `isnull().sum()`.  

Baris yang mengandung missing values diidentifikasi menggunakan `isnull().any(axis=1)` dan index-nya disimpan dalam variabel `pos_missing_train`.  

Kemudian, baris yang memiliki nilai kosong dihapus menggunakan `dropna()` sehingga hanya data lengkap yang digunakan.  
Terakhir, jumlah data setelah proses cleaning ditampilkan untuk mengetahui perubahan ukuran dataset.

**Output:**
- `train_data`: Data fitur awal (Age, Fare)
- Informasi jumlah missing values per kolom
- `pos_missing_train`: Index baris yang memiliki missing values
- `train_data_clean`: Data training yang sudah bersih
- Jumlah data setelah cleaning



Test Data (Age, Fare + Missing)

In [4]:
test_data = test_dataset[['Age', 'Fare']]

# cek missing
print(test_data.isnull().sum())

# catat posisi missing
pos_missing_test = test_data[test_data.isnull().any(axis=1)].index

# hapus missing
test_data_clean = test_data.dropna()

print("Jumlah data test setelah cleaning:", len(test_data_clean))

Age     86
Fare     1
dtype: int64
Jumlah data test setelah cleaning: 331


Penjelasan:

**Tujuan:**
Menyiapkan data testing dengan memilih fitur yang relevan serta menangani missing values agar dapat digunakan untuk evaluasi model.

**Proses:**
Kode mengambil dua fitur, yaitu `Age` dan `Fare`, dari dataset testing.  
Kemudian dilakukan pengecekan jumlah missing values pada setiap kolom menggunakan `isnull().sum()`.

Baris yang mengandung nilai kosong diidentifikasi menggunakan `isnull().any(axis=1)` dan index-nya disimpan dalam `pos_missing_test`.  

Selanjutnya, baris dengan missing values dihapus menggunakan `dropna()` untuk menghasilkan data testing yang bersih.  
Terakhir, jumlah data setelah proses cleaning ditampilkan untuk mengetahui ukuran akhir data uji.

**Output:**
- `test_data`: Data fitur awal (Age, Fare)
- Informasi jumlah missing values pada data test
- `pos_missing_test`: Index baris dengan missing values
- `test_data_clean`: Data testing yang sudah bersih
- Jumlah data test setelah cleaning



Ambil Label

In [5]:
# train label (sesuai index setelah cleaning)
train_label = dataset.loc[train_data_clean.index, 'Survived']

# test label
test_label = test_label_df.loc[test_data_clean.index, 'Survived']

Penjelasan:
**Tujuan:**
Mengambil label (target) yang sesuai dengan data training dan testing setelah proses cleaning.

**Proses:**
Kode mengambil kolom `Survived` sebagai label klasifikasi.  
Untuk data training, label diambil dari `dataset` menggunakan index yang sama dengan `train_data_clean` agar tetap selaras setelah penghapusan missing values.  

Untuk data testing, label diambil dari `test_label_df` dengan menyesuaikan index `test_data_clean`.  
Penggunaan `.loc[]` memastikan bahwa hanya baris yang sesuai dengan data bersih yang diambil.

**Output:**
- `train_label`: Label training yang sudah sesuai dengan data bersih
- `test_label`: Label testing yang sudah sesuai dengan data bersih


Normalisasi Train Data (Min-Max 0-1)


In [6]:
min_vals = train_data_clean.min()
max_vals = train_data_clean.max()

train_data_norm = (train_data_clean - min_vals) / (max_vals - min_vals)

Penjelasan:
**Tujuan:**
Melakukan normalisasi data training menggunakan metode Min-Max agar nilai fitur berada dalam rentang 0 hingga 1.

**Proses:**
Kode menghitung nilai minimum (`min_vals`) dan maksimum (`max_vals`) untuk setiap fitur (`Age` dan `Fare`) pada data training yang sudah dibersihkan.  

Selanjutnya, dilakukan transformasi menggunakan rumus Min-Max Scaling:
\[
(X - X_{min}) / (X_{max} - X_{min})
\]
Proses ini dilakukan secara otomatis oleh pandas melalui mekanisme broadcasting untuk setiap kolom.

**Output:**
- `min_vals`: Nilai minimum tiap fitur
- `max_vals`: Nilai maksimum tiap fitur
- `train_data_norm`: Data training yang telah dinormalisasi ke rentang 0–1



Normalisasi Test Data (Menggunakan min-max train)

In [7]:
test_data_norm = (test_data_clean - min_vals) / (max_vals - min_vals)

Penjelasan:
**Tujuan:**
Melakukan normalisasi data testing menggunakan parameter Min-Max yang diperoleh dari data training.

**Proses:**
Kode menerapkan rumus Min-Max Scaling pada `test_data_clean` dengan menggunakan nilai minimum (`min_vals`) dan maksimum (`max_vals`) yang sebelumnya dihitung dari data training.  

Hal ini memastikan bahwa skala data testing konsisten dengan data training, sehingga model KNN dapat menghitung jarak secara valid.

**Output:**
- `test_data_norm`: Data testing yang telah dinormalisasi dalam rentang 0–1 berdasarkan skala data training



KNN

In [8]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

error_ratios = {}

for k in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=k)

    # training
    knn.fit(train_data_norm, train_label)

    # prediksi
    pred = knn.predict(test_data_norm)

    # error ratio
    error = np.mean(pred != test_label)

    error_ratios[k] = error

# tampilkan hasil
for k, err in error_ratios.items():
    print(f"k = {k} -> Error Ratio = {err:.4f}")

k = 1 -> Error Ratio = 0.4411
k = 2 -> Error Ratio = 0.3807
k = 3 -> Error Ratio = 0.4411
k = 4 -> Error Ratio = 0.4079
k = 5 -> Error Ratio = 0.4139
k = 6 -> Error Ratio = 0.4018
k = 7 -> Error Ratio = 0.4230
k = 8 -> Error Ratio = 0.3897
k = 9 -> Error Ratio = 0.4230
k = 10 -> Error Ratio = 0.3837
k = 11 -> Error Ratio = 0.4018
k = 12 -> Error Ratio = 0.3867
k = 13 -> Error Ratio = 0.3927
k = 14 -> Error Ratio = 0.3867
k = 15 -> Error Ratio = 0.3867


Penjelasan:

**Tujuan:**
Membangun dan mengevaluasi model K-Nearest Neighbors (KNN) dengan variasi nilai k (1–15) untuk menentukan performa terbaik berdasarkan error ratio.

**Proses:**
Kode mengimpor `KNeighborsClassifier` dari sklearn dan `numpy` untuk perhitungan numerik.  
Kemudian dibuat dictionary `error_ratios` untuk menyimpan nilai error dari setiap k.

Dilakukan iterasi untuk nilai k dari 1 hingga 15:
- Model KNN dibuat dengan parameter `n_neighbors=k`
- Model dilatih menggunakan data training (`train_data_norm` dan `train_label`)
- Model melakukan prediksi terhadap data testing (`test_data_norm`)
- Hasil prediksi dibandingkan dengan label asli (`test_label`) untuk menghitung error

Error dihitung menggunakan:
\[
\text{error} = \text{mean}(pred \neq test\_label)
\]
yang berarti proporsi prediksi yang salah terhadap total data test.

Hasil error untuk setiap k disimpan dalam dictionary dan ditampilkan.

**Output:**
- `error_ratios`: Dictionary berisi pasangan nilai k dan error ratio
- Tampilan error ratio untuk setiap k dari 1 hingga 15



Hasil Klasifikasi class_result dengan test_label.

In [10]:
error_ratios = {}
error_details = {}

for k in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=k)

    # Training
    knn.fit(train_data_norm, train_label)

    # Prediksi
    class_result = knn.predict(test_data_norm)

    # Bandingkan dengan label asli
    comparison = class_result != test_label.values

    # Hitung jumlah error
    error_count = np.sum(comparison)

    # Total data test
    total_data = len(test_label)

    # Error ratio
    error_ratio = error_count / total_data

    # Simpan
    error_ratios[k] = error_ratio
    error_details[k] = {
        "error_count": error_count,
        "total_data": total_data
    }

# Tampilkan hasil
for k in range(1, 16):
    print(f"k = {k}")
    print(f"  Error Count = {error_details[k]['error_count']}")
    print(f"  Total Data  = {error_details[k]['total_data']}")
    print(f"  Error Ratio = {error_ratios[k]:.4f}")
    print("-" * 30)

k = 1
  Error Count = 146
  Total Data  = 331
  Error Ratio = 0.4411
------------------------------
k = 2
  Error Count = 126
  Total Data  = 331
  Error Ratio = 0.3807
------------------------------
k = 3
  Error Count = 146
  Total Data  = 331
  Error Ratio = 0.4411
------------------------------
k = 4
  Error Count = 135
  Total Data  = 331
  Error Ratio = 0.4079
------------------------------
k = 5
  Error Count = 137
  Total Data  = 331
  Error Ratio = 0.4139
------------------------------
k = 6
  Error Count = 133
  Total Data  = 331
  Error Ratio = 0.4018
------------------------------
k = 7
  Error Count = 140
  Total Data  = 331
  Error Ratio = 0.4230
------------------------------
k = 8
  Error Count = 129
  Total Data  = 331
  Error Ratio = 0.3897
------------------------------
k = 9
  Error Count = 140
  Total Data  = 331
  Error Ratio = 0.4230
------------------------------
k = 10
  Error Count = 127
  Total Data  = 331
  Error Ratio = 0.3837
------------------------------

Penjelasan:

**Tujuan:**
Mengevaluasi hasil klasifikasi KNN dengan membandingkan prediksi (`class_result`) terhadap label aktual (`test_label`) serta menghitung error count dan error ratio untuk setiap nilai k.

**Proses:**
Kode melakukan iterasi untuk nilai k dari 1 hingga 15:
- Model KNN dilatih menggunakan data training yang telah dinormalisasi
- Model menghasilkan prediksi (`class_result`) pada data testing
- Hasil prediksi dibandingkan dengan label asli menggunakan operasi boolean (`!=`), menghasilkan array True (salah) dan False (benar)

Jumlah error dihitung dengan menjumlahkan nilai True menggunakan `np.sum()`.  
Total data test dihitung menggunakan `len(test_label)`.  

Hasil disimpan dalam dua dictionary:
- `error_ratios`: menyimpan error ratio tiap k
- `error_details`: menyimpan detail jumlah error dan total data

Terakhir, hasil ditampilkan secara rinci untuk setiap nilai k.

**Output:**
- `error_ratios`: nilai error ratio untuk setiap k
- `error_details`: detail jumlah error dan total data
- Tampilan output berisi:
  - nilai k
  - jumlah error
  - total data test
  - error ratio
